In [1]:
from modeling.CustomDataset import * 
from modeling.model import *

/Users/augustmilliken/opt/anaconda3/envs/mlflow/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 42
torch.manual_seed(seed)
random.seed(seed) #NOTE: may be unnecessary 
torch.set_default_dtype(torch.float32) 
pd.options.mode.chained_assignment = None

In [3]:
in_date = "2017-01-01" 
train_data, dev_data, test_data, num_classes, class_weights = make_datasets(in_date, "events", seed)

Loading 2017-01-01 dataset...


In [4]:
class_weights = torch.log(class_weights + 1e-6)
class_weights = class_weights - class_weights.min()
class_weights = class_weights / class_weights.max()


In [5]:
for i, weight in enumerate(class_weights):
    if weight == 0:
        class_weights[i] = 0.01

In [6]:
in_dim = train_data.__getitem__(1)[0].shape[0] # may need to be 1
out_dim = 384 # rename
# check 8 better when streamlined
batch_size = 128#8, 16, 32, 64, 128

In [ ]:
# sampler = WeightedRandomSampler(class_weights, , replacement=True)
train_loader = DataLoader(train_data, batch_size=batch_size) #, sampler=WeightedRandomSampler(class_weights, int(train_data.__len__()*(3/4)), replacement=True)) #shuffle=True
dev_loader = DataLoader(dev_data, batch_size=batch_size) #, sampler=WeightedRandomSampler(class_weights, int(dev_data.__len__()*(3/4)), replacement=True))

In [13]:
test_loader = DataLoader(test_data, batch_size=batch_size)

In [18]:
num_heads = 2 
ff_dim = int((in_dim * (2/3)) + out_dim)
convert_drop = 0 # 0.5 for non-cosine 
sub_convert = Convertion(in_dim, out_dim, num_heads, ff_dim, convert_drop)
# sub_convert = nn.Sequential(
#     nn.Linear(in_dim, ff_dim),
#     nn.ReLU(),
#     nn.Linear(ff_dim, out_dim)
# )
convert_lr = 1e-2 
convert_decay = 1e-4
convert_optim = optim.AdamW(params=sub_convert.parameters(), lr=convert_lr, weight_decay=convert_decay) # may want to add weight decay
convert_loss_fn = nn.CosineEmbeddingLoss() 
# convert_loss_fn = nn.MSELoss()
sub_convert = fit(
    sub_convert, 
    convert_optim, 
    convert_loss_fn, 
    train_loader, 
    dev_loader, 
    num_classes, 
    5, 
    mod_to_train=4)

/Users/augustmilliken/opt/anaconda3/envs/mlflow/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(




Fitting Convertion model...
------------------------- Epoch: 1 -------------------------


Training: 100%|██████████| 6250/6250 [01:49<00:00, 57.31it/s]


Training loss: 0.40837141529083254



Validating: 100%|██████████| 782/782 [00:03<00:00, 255.50it/s]


...

Cosine similarity: 0.6005743145942688
Validation loss: 0.0017886125715449452
------------------------- Epoch: 5 -------------------------


Training: 100%|██████████| 6250/6250 [01:51<00:00, 56.13it/s]


Training loss: 0.39893780500411985



Validating: 100%|██████████| 782/782 [00:03<00:00, 255.76it/s]

Cosine similarity: 0.6005504727363586
Validation loss: 0.0017877211794257164


In [17]:
# NOTE: using if concatenating
sub_in = (out_dim + in_dim)
sub_drop = 0.3
sub_class = Classification(out_dim, num_classes, sub_drop)
class_lr = 1e-3 
class_decay = 1e-4
class_optim = optimizer = optim.AdamW(params = sub_class.parameters(), lr=class_lr, weight_decay=class_decay) 
class_loss_fn = nn.CrossEntropyLoss(weight = class_weights) 
sub_class = fit(
    model=sub_class, 
    opt=class_optim, 
    loss_fn=class_loss_fn, 
    train_data=train_loader, 
    val_data=dev_loader, 
    num_classes=num_classes,
    epochs=2, 
    mod_to_train=2)



Fitting Classification model...
------------------------- Epoch: 1 -------------------------


Training:   0%|          | 0/6250 [00:00<?, ?it/s]

Training: 100%|██████████| 6250/6250 [00:24<00:00, 258.68it/s]


Training loss: 0.02900776053600188



Validating: 100%|██████████| 782/782 [00:01<00:00, 469.99it/s]


Validation Macro precision: 0.9648085236549377
Validation Weighted precision: 0.9983077645301819
------------------------- Epoch: 2 -------------------------


Training: 100%|██████████| 6250/6250 [00:25<00:00, 249.49it/s]


Training loss: 0.001947118508598578



Validating: 100%|██████████| 782/782 [00:01<00:00, 450.40it/s]

Validation Macro precision: 0.9826155304908752
Validation Weighted precision: 0.999187707901001


In [8]:
from collections import Counter 
counter = Counter()
# print(train_data[0])
counter.update([int(label[1]) for label in test_data])
# for batch in train_data:
#     y = batch[1].detach().cpu().tolist()
#     print(y)
# counter.update(y)

In [9]:
print(counter)

Counter({6: 36838, 5: 22513, 1: 15679, 8: 9628, 2: 5450, 4: 3928, 7: 2739, 9: 1414, 10: 1128, 3: 498, 0: 185})


In [25]:
drop = 0.5
model = Multi(sub_convert, sub_class, drop, concat=False) #in_dim, sub_class, False
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

lr = 1e-6
class_decay = 1e-1

optimizer = optim.AdamW(params = model.parameters(), lr=lr, weight_decay=class_decay) 
loss_fn = nn.CrossEntropyLoss(weight = class_weights)
model = fit(model, optimizer, loss_fn, train_loader, dev_loader, num_classes, 10)

Model parameters: 2065147


Fitting Multi model...
------------------------- Epoch: 1 -------------------------


Training: 100%|██████████| 6250/6250 [01:53<00:00, 55.28it/s]


Training loss: 5.232786213531494



Validating: 100%|██████████| 782/782 [00:03<00:00, 215.16it/s]


Validation Macro precision: 0.01430272776633501
Validation Weighted precision: 0.024752728641033173
------------------------- Epoch: 2 -------------------------


...

Validation Macro precision: 0.01430272776633501
Validation Weighted precision: 0.024752728641033173
------------------------- Epoch: 10 -------------------------


Training: 100%|██████████| 6250/6250 [01:47<00:00, 58.35it/s]


Training loss: 1.7861720106124879



Validating: 100%|██████████| 782/782 [00:03<00:00, 227.37it/s]

Validation Macro precision: 0.01430272776633501
Validation Weighted precision: 0.024752728641033173


In [24]:
# classification for just game state
game_drop = 0.3
game_class = Classification(in_dim, num_classes, game_drop) 
game_lr = 1e-6
game_decay = 1e-1
game_optim = optimizer = optim.AdamW(params = game_class.parameters(), lr=game_lr, weight_decay=game_decay) 
game_loss_fn = nn.CrossEntropyLoss(weight = class_weights) # , label_smoothing=0.05 
game_class = fit(
    model=game_class, 
    opt=game_optim, 
    loss_fn=game_loss_fn, 
    train_data=train_loader, 
    val_data=dev_loader, 
    num_classes=num_classes,
    epochs=2, 
    mod_to_train=0)



Fitting Classification model...
------------------------- Epoch: 1 -------------------------


Training: 100%|██████████| 6250/6250 [00:14<00:00, 433.37it/s]


Training loss: 3212109.99958



Validating: 100%|██████████| 782/782 [00:01<00:00, 625.22it/s]


Validation Macro precision: 0.034950822591781616
Validation Weighted precision: 0.07576870173215866
------------------------- Epoch: 2 -------------------------


Training: 100%|██████████| 6250/6250 [00:14<00:00, 432.10it/s]


Training loss: 2862298.27028



Validating: 100%|██████████| 782/782 [00:01<00:00, 554.43it/s]

Validation Macro precision: 0.035210251808166504
Validation Weighted precision: 0.07641356438398361


In [10]:
for name, param in game_class.named_parameters():
    if param.grad is not None:
        print(name, param.grad)

layer1.weight tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
layer1.bias tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
layer2.weight tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [14]:
class_weights

tensor([5.0000e-01, 4.9432e-04, 6.4306e-06, 1.8263e-05, 1.9893e-04, 2.4885e-05,
        4.4315e-06, 2.7186e-06, 3.6368e-05, 1.0422e-05, 6.9585e-05])

In [17]:
class_weights

tensor([1.0000e+00, 9.8863e-04, 1.2861e-05, 3.6526e-05, 3.9785e-04, 4.9770e-05,
        8.8630e-06, 5.4372e-06, 7.2735e-05, 2.0844e-05, 1.3917e-04])

In [20]:
class_weights

tensor([1.0000e+00, 1.2189e-03, 1.5876e-05, 4.4983e-05, 4.9065e-04, 6.1447e-05,
        1.0878e-05, 6.7621e-06, 8.9670e-05, 2.5578e-05, 1.7169e-04])

In [25]:
class_weights

tensor([1.0000, 0.4142, 0.0586, 0.1393, 0.3374, 0.1643, 0.0321, 0.0100, 0.1954,
        0.0950, 0.2493])



0 other
1 single 
2 double 
3 triple 
4 homerun 
5 strikeout
6 out 
7 multi out
8 walk 
9 awarded first 
10 field 



3 double 
12 hr
18 single
(19) strike out
21 tripple 

out: 6, 8, 9, 14, 16

double play (mult out): 4, 10, 15, 17, 20, 22

walk: 13, 24

awarded first: 2, 11

other: 2, 23 (5)

field thing: 5, 7 


no strike out or walks??

In [26]:
macro_precision = Precision(task="multiclass", average="macro", num_classes=num_classes)
weighted_precision = Precision(task="multiclass", average="weighted", num_classes=num_classes)

In [35]:
convert_result = validation(sub_convert,
           test_loader,
           macro_precision,
           weighted_precision, 
           1)
print(convert_result[0])

Validating: 100%|██████████| 782/782 [00:03<00:00, 221.19it/s]

Cosine similarity: 0.5884202122688293
tensor(0.0018)


In [36]:
class_result = validation(sub_class,
           test_loader,
           macro_precision,
           weighted_precision, 
           2)
print(class_result)
macro_precision.reset()
weighted_precision.reset() 

Validating: 100%|██████████| 782/782 [00:01<00:00, 424.77it/s]

[tensor(0.9574), tensor(0.9965)]


In [33]:
model_result = validation(model,
           test_loader,
           macro_precision,
           weighted_precision, 
           0)
print(model_result)
macro_precision.reset()
weighted_precision.reset() 

Validating: 100%|██████████| 782/782 [00:03<00:00, 203.40it/s]


[tensor(0.0143), tensor(0.0246)]


In [34]:
game_result = validation(game_class,
           test_loader,
           macro_precision,
           weighted_precision, 
           0)
print(game_result)
macro_precision.reset()
weighted_precision.reset() 

Validating: 100%|██████████| 782/782 [00:01<00:00, 495.68it/s]

[tensor(0.0347), tensor(0.0751)]
